# Ham metin FULL fine-tune — XLM-R-large, hedefe doğrudan

**Amaç:** Metin tek test edilmemiş bilgi kaynağı. Mevcut metin modelleri (xlmr skalar rw=143) feature'lara koşullu + kısmi eğitimdi. Bu sürüm XLM-R-large'ı **tam unfreeze** edip hedefe **doğrudan** öğretir.

**Beklenti dürüst olarak DÜŞÜK** — metin örnekleri feature'ların özeti gibi (residual korelasyonu 0.007). Ama tek deneme bu. Karar: full-FT OOF, blend'in residual'ini açıklarsa değer; açıklamazsa metin tavanı kesin.

**Çalıştırma:** Runtime > A100 GPU. ~30-50 dk (5 fold × 6 epoch, large model).

In [ ]:
from google.colab import files
up = files.upload()  # colab_fullft_pkg.zip
import zipfile, os
z = [k for k in up if k.endswith('.zip')][0]
with zipfile.ZipFile(z) as f: f.extractall('.')
if os.path.isdir('colab_fullft_pkg'):
    for fn in os.listdir('colab_fullft_pkg'): os.replace(f'colab_fullft_pkg/{fn}', fn)
print(sorted(os.listdir('.')))

In [ ]:
!nvidia-smi -L
!pip -q install transformers sentencepiece pyarrow 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported())

## Eğitim (full fine-tune)

In [ ]:
!python -u text_fullft_core.py

## KRİTİK TEST: full-FT yeni açı katıyor mu?
Tek-model rw + blend'e ekleyince residual'i açıklıyor mu (sızıntısız nested).

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
tr = pd.read_csv('train.csv', encoding='utf-8-sig'); te = pd.read_csv('test_x.csv', encoding='utf-8-sig')
y = tr['career_success_score'].values.astype(float)
def yw(a,b,cap=(0.3,2.5)):
    pa=pd.Series(a).value_counts(normalize=True); pb=pd.Series(b).value_counts(normalize=True)
    w=pd.Series(a).map(pb/pa).fillna(1.0).clip(*cap).values; return w*len(w)/w.sum()
w = yw(tr['application_year'], te['application_year'])
def rw(p): return float(np.average((y-np.clip(p,0,100))**2,weights=w))

ft = np.load('fullft_oof.npy')
sub2 = np.load('sub2_oof.npy'); ours = np.load('ourteam_oof_tunafolds.npy')
blend = 0.5*sub2 + 0.5*ours; res = y - blend
print(f'full-FT tek-model:  duz={((y-np.clip(ft,0,100))**2).mean():.2f}  rw={rw(ft):.2f}  (xlmr-skalar 126/143)')
print(f'full-FT residual korelasyonu: {np.corrcoef(res, ft)[0,1]:+.4f}  (xlmr eski: 0.007)')
print()
# blend'e ekle: nested 3-model ridge (sub2, ours, ft)
M = np.column_stack([sub2, ours, ft])
kf = KFold(5, shuffle=True, random_state=42); pr = np.zeros(len(y))
for tri, vai in kf.split(M):
    r = Ridge(alpha=1.0, positive=True).fit(M[tri], y[tri], sample_weight=w[tri]); pr[vai] = r.predict(M[vai])
print(f'baz blend (sub2+ours)           rw = {rw(blend):.4f}')
print(f'+ full-FT (nested ridge_pos)    rw = {rw(pr):.4f}   ({rw(pr)-rw(blend):+.3f})')
print('-> negatif (düşüş) = full-FT yeni sinyal getirdi; ~0/pozitif = metin tavanı kesin')

In [ ]:
from google.colab import files
for f in ['fullft_oof.npy','fullft_test.npy']: files.download(f)